# 🔵 Clustering — Notebook Completo
> **Curso:** Machine Learning / Business Intelligence
> **Objetivo:** Descubrir estructura latente en datos sin etiquetas usando K-Means, Clustering Jerárquico y DBSCAN

---
## ¿Qué aprenderás?
1. Identificar la geometría de distintos tipos de datos
2. Preprocesar datos: codificación y escalado
3. Aplicar **K-Means**, **Clustering Jerárquico** y **DBSCAN**
4. Elegir el número óptimo de clusters (Elbow, Silhouette)
5. Evaluar calidad con métricas internas
6. Segmentar clientes bancarios reales

---
### ¿Por qué clustering?
El clustering es **aprendizaje no supervisado**: no hay etiquetas, el algoritmo descubre estructura por sí solo.

| Algoritmo | Supuesto de forma | Necesita k | Maneja ruido |
|---|---|---|---|
| K-Means | Esferas convexas | ✅ Sí | ❌ No |
| Jerárquico | Cualquiera (bottom-up) | ❌ No | ❌ No |
| DBSCAN | Cualquiera (densidad) | ❌ No | ✅ Sí |
| Mean Shift | Cualquiera (kernel) | ❌ No | Parcial |

## 1. Instalación y Configuración

In [ ]:
!pip install scikit-learn scipy matplotlib seaborn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MeanShift, estimate_bandwidth
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cdist

plt.rcParams.update({
    'figure.facecolor': '#18181b', 'axes.facecolor': '#27272a',
    'axes.edgecolor': '#3f3f46', 'text.color': '#e4e4e7',
    'axes.labelcolor': '#a1a1aa', 'xtick.color': '#71717a',
    'ytick.color': '#71717a', 'grid.color': '#3f3f46',
    'grid.alpha': 0.4, 'axes.titlesize': 12,
})

PALETTE  = ['#10b981','#3b82f6','#f59e0b','#8b5cf6','#ef4444','#ec4899']
NOISE_C  = '#52525b'
np.random.seed(42)
print('✅ Listo')

---
## 2. Datasets — Geometrías de Clustering

No todos los algoritmos funcionan bien con todos los tipos de datos. La **geometría** de los clusters importa.

| Dataset | Geometría | Algoritmo ideal |
|---|---|---|
| **Blobs** | Esferas bien separadas | K-Means |
| **Moons** | Medias lunas entrelazadas | DBSCAN |
| **Circles** | Círculos concéntricos | DBSCAN |
| **Aniso** | Elipses rotadas | K-Means falla, DBSCAN funciona |
| **Varied** | Clusters con distinta varianza | Jerárquico |
| **Noise** | Ruido uniforme | DBSCAN (identifica ruido) |

In [ ]:
N = 300

blobs,   y_blobs   = make_blobs(n_samples=N, centers=3, cluster_std=0.8, random_state=42)
moons,   y_moons   = make_moons(n_samples=N, noise=0.08, random_state=42)
circles, y_circles = make_circles(n_samples=N, factor=0.5, noise=0.07, random_state=42)

X_aniso = np.dot(make_blobs(n_samples=N, random_state=170)[0],
                 [[0.60, -0.60], [-0.40, 0.80]])
y_aniso = make_blobs(n_samples=N, random_state=170)[1]

X_varied, y_varied = make_blobs(n_samples=N, cluster_std=[0.5, 1.5, 0.3], random_state=42)

X_noise = np.random.uniform(-3, 3, size=(N, 2))
y_noise = np.zeros(N, dtype=int)

DATASETS = {
    'Blobs':   (blobs, y_blobs),
    'Moons':   (moons, y_moons),
    'Circles': (circles, y_circles),
    'Aniso':   (X_aniso, y_aniso),
    'Varied':  (X_varied, y_varied),
    'Noise':   (X_noise, y_noise),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Datasets de Clustering — Diferentes Geometrías', fontsize=14)

for ax, (name, (X, y)) in zip(axes.flat, DATASETS.items()):
    colors = [PALETTE[int(c) % len(PALETTE)] for c in y]
    ax.scatter(X[:,0], X[:,1], c=colors, s=12, alpha=0.8, edgecolors='none')
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 3. Preprocesamiento — Encoding y Escalado

Los algoritmos basados en distancias (K-Means, Jerárquico) son **muy sensibles a la escala**.

### ¿Por qué escalar?
Si `Edad` va de 20–70 e `Ingreso` de 20.000–200.000, la distancia euclideana estará **dominada por el ingreso** aunque ambas variables sean igualmente importantes.

| Método | Fórmula | Resultado | Cuándo usar |
|---|---|---|---|
| **Min-Max** | `(x - min) / (max - min)` | [0, 1] | Distribución conocida, sin outliers extremos |
| **Z-Score** | `(x - μ) / σ` | media=0, std=1 | Distribución normal aproximada |

In [ ]:
# Demostración: efecto del escalado en K-Means
np.random.seed(42)
n = 200
age    = np.random.normal(40, 10, n)           # 20–70 años
income = np.random.normal(60000, 20000, n)     # 20k–100k
X_demo = np.column_stack([age, income])

scaler_mm = MinMaxScaler()
scaler_zs = StandardScaler()
X_mm = scaler_mm.fit_transform(X_demo)
X_zs = scaler_zs.fit_transform(X_demo)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Efecto del Escalado en los Datos')

labels_raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_demo)
labels_mm  = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_mm)
labels_zs  = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_zs)

for ax, X, labels, title in [
    (axes[0], X_demo, labels_raw, 'Sin escalar\n(Ingreso domina la distancia)'),
    (axes[1], X_mm,   labels_mm,  'Min-Max Scaling\n(Ambas en [0,1])'),
    (axes[2], X_zs,   labels_zs,  'Z-Score Standardization\n(Ambas en escala σ)'),
]:
    colors = [PALETTE[l] for l in labels]
    ax.scatter(X[:,0], X[:,1], c=colors, s=15, alpha=0.8, edgecolors='none')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Edad (o escala)'); ax.set_ylabel('Ingreso (o escala)')
    sil = silhouette_score(X, labels)
    ax.text(0.02, 0.96, f'Silhouette={sil:.3f}', transform=ax.transAxes,
            fontsize=8, color='#fbbf24', va='top')

plt.tight_layout()
plt.show()
print('⚠️  Nota: sin escalar, los clusters quedan separados solo por ingreso (eje vertical).')

In [ ]:
# Encoding de variables categóricas
df_example = pd.DataFrame({
    'ciudad':         ['Santiago', 'Valparaíso', 'Concepción', 'Santiago', 'Concepción'],
    'estado_civil':   ['Soltero', 'Casado', 'Divorciado', 'Casado', 'Soltero'],
    'ingreso':        [45000, 60000, 38000, 75000, 52000],
})

print('Datos originales:')
print(df_example)
print()

# One-Hot Encoding
df_ohe = pd.get_dummies(df_example, columns=['ciudad','estado_civil'], dtype=int)
print('One-Hot Encoding (una columna por categoría):')
print(df_ohe)
print()

# Label Encoding (solo cuando hay orden natural)
le = LabelEncoder()
df_le = df_example.copy()
df_le['ciudad_code']       = le.fit_transform(df_example['ciudad'])
df_le['estado_civil_code'] = le.fit_transform(df_example['estado_civil'])
print('Label Encoding (código numérico — cuidado: impone orden artificial):')
print(df_le[['ciudad','ciudad_code','estado_civil','estado_civil_code','ingreso']])

---
## 4. K-Means

### Algoritmo (Lloyd's Algorithm)
1. Inicializar `k` centroides aleatoriamente (o con K-Means++)
2. **Asignar** cada punto al centroide más cercano
3. **Actualizar** centroides calculando la media de los puntos asignados
4. Repetir hasta convergencia

**Función objetivo:** minimizar la suma de distancias al cuadrado (WCSS/Inertia)
$$\text{WCSS} = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

In [ ]:
# Método del Codo (Elbow) — encontrar k óptimo
X_scaled = StandardScaler().fit_transform(blobs)

k_range = range(1, 11)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_ for k in k_range]
sil_scores = [silhouette_score(X_scaled, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled))
              if k > 1 else 0 for k in k_range]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Selección del k óptimo para K-Means')

axes[0].plot(list(k_range), wcss, marker='o', color='#10b981', linewidth=2)
axes[0].fill_between(list(k_range), wcss, alpha=0.15, color='#10b981')
axes[0].axvline(3, color='#f59e0b', linewidth=1.5, linestyle='--', label='codo en k=3')
axes[0].set_xlabel('Número de clusters k')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Método del Codo (Elbow Method)')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(list(k_range)[1:], sil_scores[1:], marker='s', color='#3b82f6', linewidth=2)
axes[1].fill_between(list(k_range)[1:], sil_scores[1:], alpha=0.15, color='#3b82f6')
axes[1].axvline(3, color='#f59e0b', linewidth=1.5, linestyle='--', label='máximo en k=3')
axes[1].set_xlabel('Número de clusters k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k (mayor = mejor)')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# K-Means en los 6 datasets
K_VALUES = {'Blobs': 3, 'Moons': 2, 'Circles': 2, 'Aniso': 3, 'Varied': 3, 'Noise': 3}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('K-Means en Distintas Geometrías')

for ax, (name, (X, _)) in zip(axes.flat, DATASETS.items()):
    Xs = StandardScaler().fit_transform(X)
    k  = K_VALUES[name]
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(Xs)
    sil = silhouette_score(Xs, labels) if len(set(labels)) > 1 else 0
    colors = [PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=12, alpha=0.75, edgecolors='none')
    ax.scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
               c='white', s=120, marker='*', zorder=5, edgecolors='#f59e0b', linewidths=1)
    ax.set_title(f'{name}  (k={k}, sil={sil:.2f})')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()
print('⚠️  K-Means falla en Moons y Circles (asume clusters esféricos).')
print('    Observa el silhouette score bajo para esos casos.')

---
## 5. Clustering Jerárquico

### Algoritmo (Aglomerativo — Bottom-Up)
1. Cada punto comienza como su propio cluster
2. Unir los dos clusters más cercanos
3. Repetir hasta tener un único cluster
4. **Cortar el dendrograma** a la altura que da el número de clusters deseado

El resultado se visualiza como un **dendrograma** — árbol jerárquico de fusiones.

In [ ]:
# Comparar tipos de linkage
X_hier = StandardScaler().fit_transform(blobs[:100])  # submuestra para visualización

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Dendrogramas — Comparación de Métodos de Linkage')

for ax, method, color in zip(axes, ['ward','complete','average','single'],
                              ['#10b981','#3b82f6','#f59e0b','#8b5cf6']):
    Z = linkage(X_hier, method=method)
    dendrogram(Z, ax=ax, color_threshold=0,
               above_threshold_color=color,
               leaf_font_size=0, no_labels=True)
    ax.set_title(f'Linkage: {method.capitalize()}')
    ax.set_ylabel('Distancia')
    ax.axhline(y=2.0, color='#ef4444', linewidth=1, linestyle='--', label='corte')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Cluster jerárquico en todos los datasets
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Clustering Jerárquico (Ward Linkage) en Distintas Geometrías')

N_CLUSTERS = {'Blobs': 3, 'Moons': 2, 'Circles': 2, 'Aniso': 3, 'Varied': 3, 'Noise': 4}

for ax, (name, (X, _)) in zip(axes.flat, DATASETS.items()):
    Xs = StandardScaler().fit_transform(X)
    n  = N_CLUSTERS[name]
    labels = AgglomerativeClustering(n_clusters=n, linkage='ward').fit_predict(Xs)
    sil = silhouette_score(Xs, labels)
    colors = [PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=12, alpha=0.75, edgecolors='none')
    ax.set_title(f'{name}  (n={n}, sil={sil:.2f})')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 6. DBSCAN — Density-Based Spatial Clustering

### Conceptos clave
- **ε (epsilon):** radio del vecindario alrededor de cada punto
- **min_samples:** mínimo de puntos para que una región sea densa
- **Punto core:** tiene ≥ `min_samples` vecinos en radio ε
- **Punto borde:** está dentro de ε de un core pero no es core
- **Ruido:** ni core ni borde → etiquetado como -1

**Ventaja:** encuentra clusters de **forma arbitraria** y detecta **ruido automáticamente**.

In [ ]:
# Efecto de epsilon en DBSCAN
X_moons_s = StandardScaler().fit_transform(moons)

eps_values = [0.1, 0.2, 0.3, 0.5]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Efecto de epsilon en DBSCAN (Moons, min_samples=5)')

for ax, eps in zip(axes, eps_values):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_moons_s)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = np.sum(labels == -1)
    colors = [NOISE_C if l == -1 else PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(X_moons_s[:,0], X_moons_s[:,1], c=colors, s=12, alpha=0.8, edgecolors='none')
    ax.set_title(f'ε={eps}\n{n_clusters} clusters, {n_noise} ruido', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

noise_patch = plt.matplotlib.patches.Patch(color=NOISE_C, label='Ruido (-1)')
axes[0].legend(handles=[noise_patch], fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN en todos los datasets
DBSCAN_PARAMS = {
    'Blobs': (0.4, 5), 'Moons': (0.2, 5), 'Circles': (0.2, 5),
    'Aniso': (0.3, 5), 'Varied': (0.3, 5), 'Noise': (0.15, 5),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('DBSCAN en Distintas Geometrías')

for ax, (name, (X, _)) in zip(axes.flat, DATASETS.items()):
    Xs = StandardScaler().fit_transform(X)
    eps, min_s = DBSCAN_PARAMS[name]
    labels = DBSCAN(eps=eps, min_samples=min_s).fit_predict(Xs)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = np.sum(labels == -1)
    sil = silhouette_score(Xs, labels) if len(set(labels[labels != -1])) > 1 else 0
    colors = [NOISE_C if l == -1 else PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=12, alpha=0.8, edgecolors='none')
    ax.set_title(f'{name}  ({n_clusters} clusters, {n_noise} ruido, sil={sil:.2f})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()
print('✅ DBSCAN maneja correctamente Moons, Circles y detecta ruido en Noise.')

---
## 7. Comparación de Algoritmos

¿Cuándo usar cada uno? Comparemos resultados lado a lado.

In [ ]:
# Grid: algoritmo × dataset
SELECTED_DATASETS = ['Blobs', 'Moons', 'Circles', 'Aniso']
algorithms = {
    'K-Means':      lambda X: KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X),
    'Jerárquico':   lambda X: AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X),
    'DBSCAN':       lambda X: DBSCAN(eps=0.3, min_samples=5).fit_predict(X),
}

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Comparación de Algoritmos × Geometría de Datos', fontsize=13)

for col, dname in enumerate(SELECTED_DATASETS):
    X, _ = DATASETS[dname]
    Xs = StandardScaler().fit_transform(X)
    for row, (aname, afn) in enumerate(algorithms.items()):
        ax = axes[row, col]
        labels = afn(Xs)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        sil = silhouette_score(Xs, labels) if n_clusters > 1 else 0
        colors = [NOISE_C if l == -1 else PALETTE[l % len(PALETTE)] for l in labels]
        ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=8, alpha=0.75, edgecolors='none')
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0: ax.set_title(dname, fontsize=10)
        if col == 0: ax.set_ylabel(aname, fontsize=9)
        ax.text(0.02, 0.95, f'sil={sil:.2f}', transform=ax.transAxes,
                fontsize=7, color='#fbbf24', va='top')

plt.tight_layout()
plt.show()

---
## 8. Métricas de Evaluación

### Métricas internas (sin labels verdaderas)

| Métrica | Fórmula conceptual | Rango | Mejor valor |
|---|---|---|---|
| **Silhouette** | (separación - cohesión) / max | [-1, 1] | Cercano a 1 |
| **Davies-Bouldin** | avg(max ratio dispersión/separación) | [0, ∞) | Cercano a 0 |
| **WCSS/Inertia** | Σ distancias² al centroide | [0, ∞) | Mínimo relativo (codo) |

In [ ]:
# Diagrama de Silhouette detallado
X_s = StandardScaler().fit_transform(blobs)
K_TEST = [2, 3, 4]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Diagrama de Silhouette para k=2,3,4 (dataset Blobs)')

for ax, k in zip(axes, K_TEST):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_s)
    sil_vals = silhouette_samples(X_s, labels)
    avg_sil = silhouette_score(X_s, labels)
    y_lower = 10
    for c in range(k):
        c_sil = np.sort(sil_vals[labels == c])
        size  = len(c_sil)
        y_upper = y_lower + size
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                          color=PALETTE[c], alpha=0.85)
        y_lower = y_upper + 5
    ax.axvline(avg_sil, color='#ef4444', linewidth=2, linestyle='--')
    ax.set_title(f'k={k}  |  avg silhouette={avg_sil:.3f}')
    ax.set_xlabel('Coeficiente de Silhouette')
    ax.set_ylabel('Cluster')
    ax.grid(axis='x')

plt.tight_layout()
plt.show()
print('✅ k=3 tiene el mayor silhouette promedio y las barras más uniformes → mejor clustering.')

In [ ]:
# Tabla comparativa de métricas para todos los algoritmos
print('═' * 75)
print(f"{'Algoritmo':15s} {'Dataset':10s} {'Clusters':9s} {'Silhouette':12s} {'Davies-Bouldin':16s}")
print('═' * 75)

alg_fns_table = {
    'K-Means':     lambda X, k: KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X),
    'Jerárquico':  lambda X, k: AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X),
    'DBSCAN':      lambda X, k: DBSCAN(eps=0.3, min_samples=5).fit_predict(X),
}
K_TRUE = {'Blobs': 3, 'Moons': 2, 'Circles': 2, 'Aniso': 3}

for dname in ['Blobs', 'Moons', 'Circles', 'Aniso']:
    X, _ = DATASETS[dname]
    Xs = StandardScaler().fit_transform(X)
    for aname, afn in alg_fns_table.items():
        labels = afn(Xs, K_TRUE[dname])
        n_valid = len(set(labels[labels != -1]))
        if n_valid < 2:
            print(f"{aname:15s} {dname:10s} {n_valid:9d} {'N/A':12s} {'N/A':16s}")
            continue
        mask = labels != -1
        sil = silhouette_score(Xs[mask], labels[mask])
        dbi = davies_bouldin_score(Xs[mask], labels[mask])
        print(f"{aname:15s} {dname:10s} {n_valid:9d} {sil:12.4f} {dbi:16.4f}")

print('═' * 75)

---
## 9. Caso de Estudio — Segmentación de Clientes Bancarios

Somos el equipo de analytics de un banco. Tenemos datos de **150 clientes** con variables demográficas y de comportamiento financiero. Queremos identificar **segmentos** para personalizar productos y comunicaciones.

In [ ]:
np.random.seed(42)
n_clients = 150

# Segmentos latentes:
# A: Jóvenes profesionales (25-35, ingresos medios, alto puntaje gasto, bajo ahorro)
# B: Familia consolidada (35-50, ingresos altos, puntaje gasto medio, alto ahorro)
# C: Senior conservador (55-70, ingresos medios-altos, bajo gasto, muy alto ahorro)

seg_a = int(n_clients * 0.35)
seg_b = int(n_clients * 0.40)
seg_c = n_clients - seg_a - seg_b

ages    = np.concatenate([np.random.normal(30, 4, seg_a), np.random.normal(43, 5, seg_b), np.random.normal(62, 5, seg_c)])
incomes = np.concatenate([np.random.normal(45000, 8000, seg_a), np.random.normal(80000, 15000, seg_b), np.random.normal(65000, 12000, seg_c)])
spending = np.concatenate([np.random.normal(72, 12, seg_a), np.random.normal(55, 10, seg_b), np.random.normal(30, 8, seg_c)])
savings  = np.concatenate([np.random.normal(15, 5, seg_a), np.random.normal(45, 10, seg_b), np.random.normal(65, 10, seg_c)])

true_labels = np.array([0]*seg_a + [1]*seg_b + [2]*seg_c)
cities  = np.random.choice(['Santiago', 'Valparaíso', 'Concepción', 'Temuco'], n_clients)
marital = np.random.choice(['Soltero', 'Casado', 'Divorciado'], n_clients, p=[0.35, 0.50, 0.15])

df_bank = pd.DataFrame({
    'edad':          ages.clip(18, 80).astype(int),
    'ingreso':       incomes.clip(15000, 200000).astype(int),
    'puntaje_gasto': spending.clip(0, 100).round(1),
    'ahorro_pct':    savings.clip(0, 100).round(1),
    'ciudad':        cities,
    'estado_civil':  marital,
})

print(f'Shape: {df_bank.shape}')
print()
df_bank.head(8)

In [ ]:
# EDA rápido
print(df_bank.describe().round(1))
print()
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
fig.suptitle('Distribución de Variables Numéricas — Clientes Bancarios')
for ax, col, color in zip(axes,
    ['edad','ingreso','puntaje_gasto','ahorro_pct'],
    ['#10b981','#3b82f6','#f59e0b','#8b5cf6']):
    ax.hist(df_bank[col], bins=20, color=color, alpha=0.85, edgecolor='#18181b')
    ax.set_title(col); ax.grid(axis='y')
plt.tight_layout(); plt.show()

In [ ]:
# Preprocesamiento completo
df_prep = df_bank.copy()

# One-Hot Encoding para variables categóricas
df_prep = pd.get_dummies(df_prep, columns=['ciudad', 'estado_civil'], dtype=float)

# Z-Score en numéricas
num_cols = ['edad', 'ingreso', 'puntaje_gasto', 'ahorro_pct']
scaler = StandardScaler()
df_prep[num_cols] = scaler.fit_transform(df_prep[num_cols])

print(f'Shape después del preprocesamiento: {df_prep.shape}')
print(f'Columnas: {list(df_prep.columns)}')

In [ ]:
# K-Means con k=3
X_bank = df_prep.values
km_bank = KMeans(n_clusters=3, n_init=20, random_state=42)
bank_labels = km_bank.fit_predict(X_bank)

# PCA para visualización 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_bank)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Segmentación de Clientes Bancarios — K-Means (k=3)')

# Plot PCA
colors = [PALETTE[l] for l in bank_labels]
axes[0].scatter(X_pca[:,0], X_pca[:,1], c=colors, s=30, alpha=0.8, edgecolors='none')
axes[0].set_title(f'PCA 2D — Silhouette={silhouette_score(X_bank, bank_labels):.3f}')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
for i, c in enumerate(['Segmento A', 'Segmento B', 'Segmento C']):
    axes[0].scatter([], [], c=PALETTE[i], label=c, s=50)
axes[0].legend()

# Distribución de clusters
unique, counts = np.unique(bank_labels, return_counts=True)
axes[1].bar([f'Seg {["A","B","C"][i]}' for i in unique], counts,
             color=[PALETTE[i] for i in unique], edgecolor='#18181b')
axes[1].set_title('Tamaño de Segmentos')
axes[1].set_ylabel('Número de clientes')
for x, y in zip(range(3), counts):
    axes[1].text(x, y + 0.5, str(y), ha='center', color='#e4e4e7')
axes[1].grid(axis='y')

plt.tight_layout(); plt.show()

In [ ]:
# Perfil de cada segmento
df_bank['segmento'] = bank_labels
seg_names = {0: 'Seg A', 1: 'Seg B', 2: 'Seg C'}
df_bank['segmento_nombre'] = df_bank['segmento'].map(seg_names)

profile = df_bank.groupby('segmento_nombre')[['edad','ingreso','puntaje_gasto','ahorro_pct']].mean().round(1)
print('PERFIL PROMEDIO POR SEGMENTO:')
print(profile)
print()

# Visualización de perfil en radar/parallel coordinates
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
fig.suptitle('Perfil de Cada Segmento')

for ax, col, color in zip(axes,
    ['edad', 'ingreso', 'puntaje_gasto', 'ahorro_pct'],
    ['#10b981', '#3b82f6', '#f59e0b', '#8b5cf6']):
    means = df_bank.groupby('segmento_nombre')[col].mean()
    stds  = df_bank.groupby('segmento_nombre')[col].std()
    ax.bar(means.index, means.values, color=[PALETTE[i] for i in range(3)],
           yerr=stds.values, capsize=4, edgecolor='#18181b', alpha=0.85)
    ax.set_title(col); ax.grid(axis='y')

plt.tight_layout(); plt.show()

In [ ]:
# Interpretación de negocio
print('📋 INTERPRETACIÓN DE SEGMENTOS — RESUMEN EJECUTIVO')
print('═' * 65)
for seg_id, name in seg_names.items():
    grp = df_bank[df_bank['segmento'] == seg_id]
    edad_m  = grp['edad'].mean()
    ing_m   = grp['ingreso'].mean()
    gasto_m = grp['puntaje_gasto'].mean()
    aho_m   = grp['ahorro_pct'].mean()
    n       = len(grp)
    label = (
        '🟢 Jóvenes Profesionales' if edad_m < 40 else
        '🔵 Familia Consolidada'   if edad_m < 55 else
        '🟡 Senior Conservador'
    )
    print(f"\n  {name} — {label} (n={n})")
    print(f"    Edad: {edad_m:.0f}a  |  Ingreso: ${ing_m:,.0f}")
    print(f"    Gasto: {gasto_m:.0f}/100  |  Ahorro: {aho_m:.0f}%")
    action = (
        '→ Crédito de consumo, tarjetas de beneficios, cuenta corriente digital' if edad_m < 40 else
        '→ Crédito hipotecario, seguros de vida, fondo de educación hijos'        if edad_m < 55 else
        '→ Fondos mutuos conservadores, seguro de salud, planificación patrimonial'
    )
    print(f"    Productos recomendados: {action}")
print()

---
## 10. Ejercicios

### 🔵 Ejercicio 1 — Algoritmo incorrecto para la geometría
Aplica K-Means con k=2 al dataset **Circles**. ¿Qué silhouette score obtienes? Compara con DBSCAN. ¿Por qué K-Means falla aquí?

### ⚙️ Ejercicio 2 — Sensibilidad al epsilon en DBSCAN
Para el dataset **Moons**, prueba ε = 0.05, 0.15, 0.30, 0.50. Grafica el número de clusters y puntos de ruido detectados para cada valor. ¿Cuál es el rango óptimo de ε?

### 📏 Ejercicio 3 — Importancia del escalado
Aplica K-Means al dataset bancario **sin** escalar. Compara el silhouette score y el perfil de segmentos con la versión escalada. ¿Cuál variable domina cuando no se escala?

### 🌲 Ejercicio 4 — Corte del dendrograma
Para el dataset bancario, construye un dendrograma (Ward linkage). ¿A qué altura cortarías para obtener 3 clusters? ¿Y para 4? Visualiza ambas soluciones.

### 💼 Ejercicio 5 — Nuevo dominio
Cambia el caso de estudio a una **tienda de e-commerce**: variables serían `dias_desde_ultima_compra`, `frecuencia_compras`, `valor_promedio_pedido` (análisis RFM clásico). Genera datos sintéticos y repite el análisis completo.